In [1]:
!pip -q install --no-cache-dir  "numpy==2.0.2"  "pandas==2.2.2" "scikit-learn==1.6.1" \
"tqdm==4.67.1" "safetensors==0.5.3" \
"huggingface-hub==0.30.2"  "tokenizers==0.21.1" "transformers==4.51.3" \
"accelerate==1.6.0"  "bitsandbytes==0.48.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 216.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 305.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 350.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 323.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 237.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 354.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 134.0 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which i

## config

In [2]:
import re, gc
import json
import shutil
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Chỉ đổi biến này khi chuyển từ dev sang test.
RUN_TAG = "test"  # "dev" hoặc "test"

DATASET_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/datasett/dev_dataset.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/datasett/test_dataset.jsonl",
}

RETRIEVED_EVIDENCE_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/retrieval-output-dev/hybrid_reranker_alpha05_top10_all_labels_dev.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/retrieval-output-test/hybrid_reranker_alpha05_top10_all_labels_test.jsonl",
}

CORPUS_PATH = "/kaggle/input/datasets/lanthanhvi/corpus/full_DVSK_data_aiteamvn_256.json"
OUTPUT_DIR = Path("/kaggle/working/datasets/lanthanhvi/verification_outputss")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {"tag": "qwen25_3b_full", "name": "Qwen/Qwen2.5-3B-Instruct", "quantization": None},
    {"tag": "qwen25_7b_8bit", "name": "Qwen/Qwen2.5-7B-Instruct", "quantization": "8bit"},
]

SETTINGS = ["claim_only", "gold_evidence", "retrieved_evidence"]
# SETTINGS = ["gold_evidence"]

LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"]
EXTENDED_LABELS = LABELS + ["PARSE_ERROR"]

MAX_EVIDENCE_ITEMS = 5
MAX_SAMPLES = None  # Đặt None khi chạy chính thức.
FORCE_RERUN = False


from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
secret_label = "HF_ViHiFact"

HF_TOKEN = UserSecretsClient().get_secret(secret_label)
HF_REPO_ID = "ViLannnn/ViHisFact"

api = HfApi() 

In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def read_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


dataset = read_jsonl(DATASET_PATHS[RUN_TAG])
if MAX_SAMPLES is not None:
    dataset = dataset[:MAX_SAMPLES]

retrieval_rows = read_jsonl(RETRIEVED_EVIDENCE_PATHS[RUN_TAG])
retrieved_map = {str(row["claim_id"]): row for row in retrieval_rows}

corpus = read_json_list(CORPUS_PATH)
corpus_by_id = {str(row["chunk_id"]): row for row in corpus}

print("RUN_TAG:", RUN_TAG)
print("Dataset size:", len(dataset))
print("Retrieved map size:", len(retrieved_map))
print("Corpus size:", len(corpus_by_id))
print("\nLabel distribution:")
print(pd.Series([row["label"] for row in dataset]).value_counts())
print("\nClaim type distribution:")
print(pd.Series([row["claim_type"] for row in dataset]).value_counts())

RUN_TAG: test
Dataset size: 1050
Retrieved map size: 1050
Corpus size: 5343

Label distribution:
SUPPORTED              350
NOT_ENOUGH_EVIDENCE    350
REFUTED                350
Name: count, dtype: int64

Claim type distribution:
MILITARY_DIPLOMACY          357
PERSON_ROLE                 254
POLITICAL_ADMINISTRATIVE    159
TIME_EVENT                  151
GENEALOGY_SUCCESSION         81
NATURAL_OMEN_DISASTER        48
Name: count, dtype: int64


## Format evidence và tạo prompt

- Gold evidence: lấy `evidence_ids` rồi tra `raw_text` tương ứng trong corpus.
- Retrieved evidence: lấy `retrieved_evidence[*].chunk_id` rồi tra `raw_text` tương ứng trong corpus.

## Parse output và gọi LLM

In [4]:
def get_raw_texts_from_chunk_ids(chunk_ids):
    texts = []
    for chunk_id in chunk_ids:
        chunk = corpus_by_id.get(str(chunk_id))
        if chunk is None:
            raise KeyError(f"Không tìm thấy chunk_id trong corpus: {chunk_id}")
        raw_text = str(chunk.get("raw_text", "")).strip()
        if not raw_text:
            raise ValueError(f"Chunk không có raw_text hợp lệ: {chunk_id}")
        texts.append(raw_text)
    return texts


def get_retrieval_info(item):
    retrieval_row = retrieved_map[str(item["claim_id"])]
    evidences = retrieval_row["retrieved_evidence"][:MAX_EVIDENCE_ITEMS]
    gold_evidence_ids = {str(x) for x in retrieval_row.get("evidence_ids", item["evidence_ids"])}
    retrieved_chunk_ids = [str(ev["chunk_id"]) for ev in evidences]

    gold_rank = None
    for ev in evidences:
        if str(ev["chunk_id"]) in gold_evidence_ids:
            gold_rank = int(ev["rank"])
            break

    return {
        "gold_evidence_ids": sorted(gold_evidence_ids),
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "retrieval_hit": gold_rank is not None,
        "gold_rank": gold_rank,
    }


def format_retrieved_evidence(item):
    retrieval_row = retrieved_map[str(item["claim_id"])]
    evidences = retrieval_row["retrieved_evidence"][:MAX_EVIDENCE_ITEMS]
    blocks = []

    for ev in evidences:
        chunk_id = str(ev["chunk_id"])
        raw_text = get_raw_texts_from_chunk_ids([chunk_id])[0]
        blocks.append(f"Evidence {ev['rank']}:\n{raw_text}")

    return "\n\n".join(blocks)




def build_prompt(item, setting):
    claim = item["claim"].strip()

    if setting == "claim_only":
        return f"""Bạn là hệ thống kiểm chứng phát biểu lịch sử Việt Nam.
    
    Dựa trên kiến thức lịch sử của bạn, hãy phân loại CLAIM vào đúng một trong ba nhãn:
    
    - SUPPORTED: Các nội dung có thể kiểm chứng trong CLAIM phù hợp với kiến thức lịch sử đã biết và không có chi tiết trọng tâm nào sai.
    - REFUTED: Có ít nhất một chi tiết trọng tâm trong CLAIM mâu thuẫn với kiến thức lịch sử đã biết.
    - NOT_ENOUGH_EVIDENCE: Bạn không có đủ kiến thức để xác định CLAIM là đúng hay sai.
    
    Quy tắc:
    - Đánh giá toàn bộ CLAIM, bao gồm chủ thể, thời gian, địa điểm, quan hệ, hành động và kết quả nếu các yếu tố này xuất hiện.
    - Chỉ chọn SUPPORTED khi tất cả chi tiết trọng tâm đều đúng.
    - Chọn REFUTED khi xác định được một chi tiết trọng tâm sai; trong phần giải thích, nêu rõ chi tiết đó.
    - Chọn NOT_ENOUGH_EVIDENCE khi không thể xác nhận cũng không thể bác bỏ chi tiết quyết định của CLAIM.
    - Không tự bổ sung dữ kiện không có trong CLAIM.
    - Chỉ trả về một đối tượng JSON hợp lệ, không dùng Markdown và không thêm nội dung bên ngoài JSON.
    
    CLAIM:
    {claim}
    

Định dạng đầu ra:
{{"label":"SUPPORTED hoặc REFUTED hoặc NOT_ENOUGH_EVIDENCE","explanation":"Giải thích ngắn gọn trong tối đa hai câu."}}"""
        
    if setting == "gold_evidence":
        evidence_text = "\n\n".join(get_raw_texts_from_chunk_ids(item["evidence_ids"]))
    elif setting == "retrieved_evidence":
        evidence_text = format_retrieved_evidence(item)
    else:
        raise ValueError(f"Unknown setting: {setting}")

    return f"""Bạn là hệ thống kiểm chứng phát biểu lịch sử.
Hãy xác định quan hệ giữa CLAIM và EVIDENCE chỉ dựa trên bằng chứng được cung cấp. Không sử dụng kiến thức bên ngoài và không suy diễn vượt quá nội dung bằng chứng.

Nhãn:
- SUPPORTED: EVIDENCE cung cấp đủ thông tin để xác nhận tất cả nội dung trọng tâm của CLAIM.
- REFUTED: EVIDENCE trực tiếp mâu thuẫn với ít nhất một nội dung trọng tâm có vai trò quyết định trong CLAIM.
- NOT_ENOUGH_EVIDENCE: EVIDENCE không cung cấp đủ thông tin để xác nhận hoặc bác bỏ CLAIM. Việc EVIDENCE không đề cập đến một chi tiết không đồng nghĩa với việc chi tiết đó bị phản bác.

Quy tắc:
- Kiểm tra các yếu tố trọng tâm như nhân vật, thời gian, địa điểm, chức vụ, quan hệ, hành động và kết quả.
- Chỉ chọn SUPPORTED khi tất cả nội dung trọng tâm được xác nhận.
- Chọn REFUTED khi có mâu thuẫn trực tiếp và rõ ràng.
- Chọn NOT_ENOUGH_EVIDENCE khi bằng chứng thiếu thông tin cần thiết hoặc chỉ liên quan chung đến chủ đề.
- Không chọn SUPPORTED chỉ vì CLAIM và EVIDENCE có nhiều từ hoặc thực thể giống nhau.
- Chỉ trả về đúng một JSON hợp lệ, không thêm Markdown hoặc văn bản bên ngoài JSON.

CLAIM:
{claim}

EVIDENCE:
{evidence_text}

Định dạng đầu ra:
{{"label":"SUPPORTED hoặc REFUTED hoặc NOT_ENOUGH_EVIDENCE","explanation":"Giải thích ngắn gọn trong tối đa hai câu."}}"""

In [5]:
def parse_verification_output(raw_text):
    text = (raw_text or "").strip()
    candidates = []

    # 1. JSON trong code block
    fence = re.search(
        r"```(?:json)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )
    if fence:
        candidates.append(fence.group(1).strip())

    # 2. Toàn bộ text có thể là JSON
    candidates.append(text)

    # 3. JSON object nằm trong text
    json_obj = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if json_obj:
        candidates.append(json_obj.group(0).strip())

    last_label = ""
    last_explanation = ""

    for cand in candidates:
        try:
            # Sửa lỗi model trả thừa một dấu " ngay trước dấu }
            cand = re.sub(r'""\s*}', '"}', cand)

            obj = json.loads(cand)

            label = str(obj.get("label", "")).strip().upper()
            explanation = str(obj.get("explanation", "")).strip()

            last_label = label
            last_explanation = explanation

            if label not in LABELS:
                continue

            if not explanation:
                return label, "", "missing_explanation"

            return label, explanation, None

        except Exception:
            continue

    return (
        "PARSE_ERROR",
        last_explanation,
        f"parse_error_or_invalid_label: {last_label}"
    )


def generate_one(tokenizer, model, prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = inputs.to(input_device)
    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=None,    
            top_p=None,         
            top_k=None,         
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0, input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Tải tokenizer và model

In [6]:
def get_quant_config(quantization):
    if quantization is None:
        return None
    if quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    raise ValueError(f"Unsupported quantization: {quantization}")


def load_llm(model_name, quantization):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    load_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
        "low_cpu_mem_usage": True,
    }

    quant_config = get_quant_config(quantization)
    if quant_config is not None:
        load_kwargs["quantization_config"] = quant_config

    model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
    model.eval()
    return tokenizer, model

## Chạy dự đoán theo từng model và setting

In [7]:
def run_one_setting(tokenizer, model, model_tag, quantization, setting):
    rows = []

    for item in tqdm(dataset, desc=f"{model_tag} | {setting}"):
        prompt = build_prompt(item, setting)
        raw_response = generate_one(tokenizer, model, prompt)
        predicted_label, predicted_explanation, parse_error = parse_verification_output(raw_response)

        row = {
            "claim_id": str(item["claim_id"]),
            "claim": item["claim"],
            "claim_type": item["claim_type"],
            "gold_label": item["label"],
            "predicted_label": predicted_label,
            "predicted_explanation": predicted_explanation,
            "raw_response": raw_response,
            "parse_error": parse_error is not None,
            "parse_error_message": parse_error,
            "model_tag": model_tag,
            "quantization": quantization or "full_fp16",
            "setting": setting,
        }

        if setting == "retrieved_evidence":
            row.update(get_retrieval_info(item))

        rows.append(row)

    return rows

In [8]:
all_predictions = []

for cfg in MODELS:
    print("\n" + "=" * 80)
    print("Model:", cfg)

    pending_settings = []
    for setting in SETTINGS:
        out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"
        if out_path.exists() and not FORCE_RERUN:
            print("Skip existing:", out_path)
            rows = read_jsonl(out_path)
            all_predictions.extend(rows)
        else:
            pending_settings.append(setting)

    if not pending_settings:
        continue

    tokenizer = None
    model = None

    try:
        tokenizer, model = load_llm(cfg["name"], cfg["quantization"])
        print("Device map:", getattr(model, "hf_device_map", None))

        for setting in pending_settings:
            out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"
            rows = run_one_setting(
                tokenizer=tokenizer,
                model=model,
                model_tag=cfg["tag"],
                quantization=cfg["quantization"],
                setting=setting,
            )
            save_jsonl(rows, out_path)

            try:
                api.upload_file(
                    path_or_fileobj=str(out_path),
                    path_in_repo=out_path.name,
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    token=HF_TOKEN,
                )
                print("Uploaded to Hugging Face successfully:", out_path.name)
            except Exception as e:
                print("Upload failed:", e)
                
            print("Saved:", out_path)
            all_predictions.extend(rows)

    finally:
        print("Cleaning up model:", cfg["tag"])
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        model = None
        tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

all_path = OUTPUT_DIR / f"{RUN_TAG}_verification_predictions_all.jsonl"
save_jsonl(all_predictions, all_path)

print("Saved all:", all_path)
print("\n Total predictions:", len(all_predictions))


Model: {'tag': 'qwen25_3b_full', 'name': 'Qwen/Qwen2.5-3B-Instruct', 'quantization': None}
Skip existing: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_3b_full_claim_only_predictions.jsonl
Skip existing: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_3b_full_gold_evidence_predictions.jsonl
Skip existing: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_3b_full_retrieved_evidence_predictions.jsonl

Model: {'tag': 'qwen25_7b_8bit', 'name': 'Qwen/Qwen2.5-7B-Instruct', 'quantization': '8bit'}
Skip existing: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_7b_8bit_claim_only_predictions.jsonl
Skip existing: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_7b_8bit_gold_evidence_predictions.jsonl


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


qwen25_7b_8bit | retrieved_evidence:   0%|          | 0/1050 [00:00<?, ?it/s]

Uploaded to Hugging Face successfully: test_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
Saved: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
Cleaning up model: qwen25_7b_8bit
Saved all: /kaggle/working/datasets/lanthanhvi/verification_outputss/test_verification_predictions_all.jsonl

 Total predictions: 6300


## Metric tổng hợp

In [9]:
def compute_metric_row(group):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    per_label = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)

    return {
        "model_tag": group["model_tag"].iloc[0],
        # "model_name": group["model_name"].iloc[0],
        # "quantization": group["quantization"].iloc[0],
        "setting": group["setting"].iloc[0],
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "f1_supported": per_label[0],
        "f1_refuted": per_label[1],
        "f1_not_enough_evidence": per_label[2],
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        # "missing_retrieval_count": int(group["missing_retrieval"].fillna(False).sum()),
    }


df = pd.DataFrame(all_predictions)
if df.empty:
    raise ValueError("No predictions found. Run prediction cells first.")

summary_rows = []
for _, group in df.groupby(["model_tag", "setting"], sort=False):
    summary_rows.append(compute_metric_row(group))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"{RUN_TAG}_verification_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df

,model_tag,setting,num_samples,accuracy,macro_f1,f1_supported,f1_refuted,f1_not_enough_evidence,parse_error_count
0,qwen25_3b_full,claim_only,1050,0.360000,0.260118,0.065163,0.210046,0.505146,0
1,qwen25_3b_full,gold_evidence,1050,0.464762,0.386423,0.548957,0.060942,0.549372,2
2,qwen25_3b_full,retrieved_evidence,1050,0.348571,0.199785,0.086022,0.005698,0.507636,2
3,qwen25_7b_8bit,claim_only,1050,0.360000,0.324677,0.139130,0.462671,0.372230,12
4,qwen25_7b_8bit,gold_evidence,1050,0.650476,0.635424,0.713514,0.550669,0.642089,1
5,qwen25_7b_8bit,retrieved_evidence,1050,0.618095,0.612310,0.688053,0.570481,0.578397,19


## Confusion matrix

In [10]:
cm_rows = []

for (model_tag, setting), group in df.groupby(["model_tag", "setting"], sort=False):
    cm = confusion_matrix(group["gold_label"], group["predicted_label"], labels=LABELS)
    for i, true_label in enumerate(LABELS):
        for j, pred_label in enumerate(LABELS):
            cm_rows.append({
                "model_tag": model_tag,
                "setting": setting,
                "true_label": true_label,
                "predicted_label": pred_label,
                "count": int(cm[i, j]),
            })

cm_df = pd.DataFrame(cm_rows)
cm_path = OUTPUT_DIR / f"{RUN_TAG}_verification_confusion_matrix.csv"
cm_df.to_csv(cm_path, index=False)
cm_df.head(30)

,model_tag,setting,true_label,predicted_label,count
0,qwen25_3b_full,claim_only,SUPPORTED,SUPPORTED,13
1,qwen25_3b_full,claim_only,SUPPORTED,REFUTED,37
2,qwen25_3b_full,claim_only,SUPPORTED,NOT_ENOUGH_EVIDENCE,300
3,qwen25_3b_full,claim_only,REFUTED,SUPPORTED,10
4,qwen25_3b_full,claim_only,REFUTED,REFUTED,46
5,qwen25_3b_full,claim_only,REFUTED,NOT_ENOUGH_EVIDENCE,294
6,qwen25_3b_full,claim_only,NOT_ENOUGH_EVIDENCE,SUPPORTED,26
7,qwen25_3b_full,claim_only,NOT_ENOUGH_EVIDENCE,REFUTED,5
8,qwen25_3b_full,claim_only,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,319
9,qwen25_3b_full,gold_evidence,SUPPORTED,SUPPORTED,171


## Metric theo claim_type

In [11]:
claim_type_rows = []

for (model_tag, setting, claim_type), group in df.groupby(["model_tag", "setting", "claim_type"], sort=False):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    claim_type_rows.append({
        "model_tag": model_tag,
        "setting": setting,
        "claim_type": claim_type,
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    })

claim_type_df = pd.DataFrame(claim_type_rows)
claim_type_path = OUTPUT_DIR / f"{RUN_TAG}_verification_claim_type_metrics.csv"
claim_type_df.to_csv(claim_type_path, index=False)
claim_type_df.head(30)

,model_tag,setting,claim_type,num_samples,accuracy,macro_f1,parse_error_count
0,qwen25_3b_full,claim_only,TIME_EVENT,151,0.655629,0.362479,0
1,qwen25_3b_full,claim_only,PERSON_ROLE,254,0.283465,0.189421,0
2,qwen25_3b_full,claim_only,GENEALOGY_SUCCESSION,81,0.209877,0.177519,0
3,qwen25_3b_full,claim_only,POLITICAL_ADMINISTRATIVE,159,0.383648,0.240645,0
4,qwen25_3b_full,claim_only,MILITARY_DIPLOMACY,357,0.324930,0.272918,0
5,qwen25_3b_full,claim_only,NATURAL_OMEN_DISASTER,48,0.270833,0.161234,0
6,qwen25_3b_full,gold_evidence,TIME_EVENT,151,0.609272,0.390372,0
7,qwen25_3b_full,gold_evidence,PERSON_ROLE,254,0.452756,0.389375,1
8,qwen25_3b_full,gold_evidence,GENEALOGY_SUCCESSION,81,0.407407,0.330270,0
9,qwen25_3b_full,gold_evidence,POLITICAL_ADMINISTRATIVE,159,0.515723,0.411508,0


## Phân tích retrieved_evidence theo retrieval_hit

Phần này chỉ áp dụng cho setting `retrieved_evidence`.

- `retrieval_hit = True`: top-5 retrieved evidence có chứa ít nhất một `gold_evidence_ids`.
- `gold_rank`: vị trí đầu tiên của gold evidence trong top-5.

Nếu `retrieval_hit=True` nhưng verifier sai, lỗi nghiêng về verifier/prompt/model. Nếu `retrieval_hit=False` và verifier sai, lỗi có thể bắt nguồn từ retrieval.

In [12]:
retrieved_df = df[df["setting"] == "retrieved_evidence"].copy()

retrieval_hit_rows = []
if not retrieved_df.empty:
    for (model_tag, retrieval_hit), group in retrieved_df.groupby(["model_tag", "retrieval_hit"], dropna=False, sort=False):
        y_true = group["gold_label"].astype(str).tolist()
        y_pred = group["predicted_label"].astype(str).tolist()
        retrieval_hit_rows.append({
            "model_tag": model_tag,
            "retrieval_hit": retrieval_hit,
            "num_samples": len(group),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
            "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        })

retrieval_hit_df = pd.DataFrame(retrieval_hit_rows)
retrieval_hit_path = OUTPUT_DIR / f"{RUN_TAG}_verification_by_retrieval_hit.csv"
retrieval_hit_df.to_csv(retrieval_hit_path, index=False)
retrieval_hit_df

,model_tag,retrieval_hit,num_samples,accuracy,macro_f1,parse_error_count
0,qwen25_3b_full,True,990,0.335354,0.195092,2
1,qwen25_3b_full,False,60,0.566667,0.241135,0
2,qwen25_7b_8bit,True,990,0.625253,0.617388,19
3,qwen25_7b_8bit,False,60,0.500000,0.438206,0


In [13]:
# File chẩn đoán theo từng claim cho setting retrieved_evidence.
retrieval_diag_cols = [
    "model_tag", "claim_id", "gold_label", "predicted_label", "claim_type",
    "gold_evidence_ids", "retrieved_chunk_ids", "retrieval_hit", "gold_rank", "parse_error"
]

retrieval_diag_df = retrieved_df[retrieval_diag_cols].copy() if not retrieved_df.empty else pd.DataFrame(columns=retrieval_diag_cols)
retrieval_diag_path = OUTPUT_DIR / f"{RUN_TAG}_verification_retrieval_diagnostics.csv"
retrieval_diag_df.to_csv(retrieval_diag_path, index=False)
retrieval_diag_df.head(20)

,model_tag,claim_id,gold_label,predicted_label,claim_type,gold_evidence_ids,retrieved_chunk_ids,retrieval_hit,gold_rank,parse_error
2100,qwen25_3b_full,claim_000001,SUPPORTED,NOT_ENOUGH_EVIDENCE,TIME_EVENT,[019f17e0-fc5a-7cf2-ba14-b20bc5b39825],"[019f17e0-fc5a-7cf2-ba14-b20bc5b39825, 019f17e...",True,1.0,False
2101,qwen25_3b_full,claim_000003,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,TIME_EVENT,[019f17e0-fc5a-7cf2-ba14-b20bc5b39825],"[019f17e2-91a0-74bc-97d8-2e5dfd2e9703, 019f17e...",True,3.0,False
2102,qwen25_3b_full,claim_000004,SUPPORTED,NOT_ENOUGH_EVIDENCE,PERSON_ROLE,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False
2103,qwen25_3b_full,claim_000005,REFUTED,NOT_ENOUGH_EVIDENCE,GENEALOGY_SUCCESSION,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False
2104,qwen25_3b_full,claim_000006,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,TIME_EVENT,[019f17e0-fc85-73fc-b591-3981c10946ae],"[019f17e0-fc85-73fc-b591-3981c10946ae, 019f17e...",True,1.0,False
2105,qwen25_3b_full,claim_000007,SUPPORTED,NOT_ENOUGH_EVIDENCE,POLITICAL_ADMINISTRATIVE,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fd57-7f7e-8051-5da239cf14fd, 019f17e...",True,1.0,False
2106,qwen25_3b_full,claim_000008,REFUTED,NOT_ENOUGH_EVIDENCE,POLITICAL_ADMINISTRATIVE,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fd57-7f7e-8051-5da239cf14fd, 019f17e...",True,1.0,False
2107,qwen25_3b_full,claim_000009,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,POLITICAL_ADMINISTRATIVE,[019f17e0-fd57-7f7e-8051-5da239cf14fd],"[019f17e0-fd57-7f7e-8051-5da239cf14fd, 019f17e...",True,1.0,False
2108,qwen25_3b_full,claim_000010,SUPPORTED,NOT_ENOUGH_EVIDENCE,POLITICAL_ADMINISTRATIVE,[019f17e0-fefe-74be-b3cc-0e4cab13c8bd],"[019f17e0-fefe-74be-b3cc-0e4cab13c8bd, 019f17e...",True,1.0,False
2109,qwen25_3b_full,claim_000011,REFUTED,NOT_ENOUGH_EVIDENCE,MILITARY_DIPLOMACY,[019f17e0-fefe-74be-b3cc-0e4cab13c8bd],"[019f17e0-fefe-74be-b3cc-0e4cab13c8bd, 019f17e...",True,1.0,False


## Nén output

In [14]:
zip_base = Path(f"/kaggle/working/verification_outputs_clean_{RUN_TAG}")
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("ZIP:", zip_path)
print("Files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

ZIP: /kaggle/working/verification_outputs_clean_test.zip
Files:
- dev_qwen25_3b_full_claim_only_predictions.jsonl
- dev_qwen25_3b_full_gold_evidence_predictions.jsonl
- dev_qwen25_3b_full_retrieved_evidence_predictions.jsonl
- dev_qwen25_7b_8bit_claim_only_predictions.jsonl
- dev_qwen25_7b_8bit_gold_evidence_predictions.jsonl
- dev_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
- dev_verification_by_retrieval_hit.csv
- dev_verification_claim_type_metrics.csv
- dev_verification_confusion_matrix.csv
- dev_verification_predictions_all.jsonl
- dev_verification_retrieval_diagnostics.csv
- dev_verification_summary.csv
- test_qwen25_3b_full_claim_only_predictions.jsonl
- test_qwen25_3b_full_gold_evidence_predictions.jsonl
- test_qwen25_3b_full_retrieved_evidence_predictions.jsonl
- test_qwen25_7b_8bit_claim_only_predictions.jsonl
- test_qwen25_7b_8bit_gold_evidence_predictions.jsonl
- test_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
- test_verification_by_retrieval_hit.csv
- test